# Tutorial: My first agent

The instructions for this tutorial can be found in the [documentation](https://metasmith.readthedocs.io/en/latest/tutorials/my_first_agent.html).

In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Runtime
from metasmith.python_api import DataTypeLibrary, DataInstanceLibrary, TransformInstanceLibrary
from metasmith.python_api import Source, Logistics
from metasmith.python_api import TargetBuilder, Resources, Size, Duration
from metasmith.python_api import ipynbButtonLink

WORKSPACE = Path("../../").resolve() # back twice since we are in example_resources/tutorials
MLIB = WORKSPACE/"MetasmithLibraries"
WORKSPACE

In [ ]:
agent_home = Source.FromLocal(WORKSPACE/"msm_home")
smith = Agent(
    home = agent_home,
    runtime=Runtime.DOCKER,
)

smith.Deploy()

In [ ]:
inputs_path = WORKSPACE/"3pangenome.xgdb"

try:
    inputs = DataInstanceLibrary.Load(inputs_path)
except:
    inputs = DataInstanceLibrary(inputs_path)
    inputs.Purge()
    inputs.AddTypeLibrary(MLIB/"data_types/ncbi.yml")
    inputs.AddTypeLibrary(MLIB/"data_types/sequences.yml")
    inputs.AddTypeLibrary(MLIB/"data_types/pangenome.yml")

    group = inputs.AddValue("pangenome", "e coli", "pangenome::pangenome")
    inputs.AddValue("DH10b",  "GCF_000019425.1", "ncbi::assembly_accession", parents={group})
    inputs.AddValue("K12",    "GCF_000005845.2", "ncbi::assembly_accession", parents={group})
    inputs.AddValue("EPI300", "GCF_049667475.1", "ncbi::assembly_accession", parents={group})
    inputs.Save()


In [ ]:
resources = [
    DataInstanceLibrary.Load(MLIB/f"resources/{n}")
    for n in ["containers", "lib"]
]

transforms = [
    TransformInstanceLibrary.Load(MLIB/f"transforms/{n}")
    for n in ["logistics", "pangenome"]
]

targets = TargetBuilder()
targets.Add("pangenome::heatmap")

task = smith.GenerateWorkflow(
    samples=inputs.AsSamples("ncbi::assembly_accession"),
    resources=resources,
    transforms=transforms,
    targets=targets,
)

In [ ]:
print(f'this workflow is called [{len(task.GetKey())}]')

In [ ]:
print(f'generated plan has [{len(task.plan.steps)}] steps')

workflow_diagram_path = f"{task.GetKey()}.dag.svg"
task.plan.RenderDAG(workflow_diagram_path)
print(f'diagram at [{workflow_diagram_path}]')

ipynbButtonLink(f"{workflow_diagram_path}", "view workflow diagram")

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [ ]:
smith.RunWorkflow(
    task,
    config_file=smith.GetNxfConfigPresets()["local"],
    resource_overrides={
        "all": Resources(
            memory=Size.GB(2),
        )
    }
)

In [ ]:
smith.WaitForWorkflow(task)

<div class="alert alert-info"><strong>Running this as a plain Python script?</strong> <code>RunWorkflow</code> launches Nextflow in the background and returns immediately. In a notebook, just advance to the next cell after the run finishes. In a <code>.py</code> script, wait for the <code>"run completed at"</code> sentinel in <code>runs/&lt;key&gt;/_metasmith/logs.latest/agent.log</code> before calling <code>CheckWorkflow</code> or <code>GetResultSource</code>. See the <em>Python basics</em> tutorial for a ready-made <code>wait_for_run</code> helper.</div>

In [ ]:
smith.CheckWorkflow(task)

In [ ]:
results_path = smith.GetResultSource(task).GetPath()
results = DataInstanceLibrary.Load(results_path)

In [ ]:
ipynbButtonLink(results_path/"_metadata/logs.latest/nxf_report.html")
ipynbButtonLink(results_path/"_metadata/logs.latest/nxf_timeline.html")

for path, type_name, endpoint in results.Iterate():
    if path.is_absolute(): continue # inputs have absolute paths
    ipynbButtonLink(results_path/path, f'view {type_name} {path.name}')